# Session 13 - Debug -> Trace -> Test
## Group C - The Average Grade (Logic Error)

Work through the cells top to bottom (**Kernel -> Restart & Run All**). Every phase of the activity sheet is covered:

1. Run & Classify (no AI)
2. Trace table
3. AI diagnosis & fix (Prompts 1-3)
4. Prove the fix - test design table, `unittest` suite, proof it fails on the buggy code
5. AI-generated tests, gap analysis (Prompts 5-7) and debrief answers
6. GitHub / Claude Code commit messages

---
## Phase 1 - Run & Classify (no AI)

The original code, exactly as given. It is renamed `average_grade_buggy` so we can keep it alongside the fixed version and test both.

In [ ]:
def average_grade_buggy(scores):
    total = 0
    for s in scores:
        total = s
    return total / len(scores)

print(average_grade_buggy([80, 70, 90, 60]))

**Symptom:** the program does **not** crash. It prints `15.0`.

**What it SHOULD return:** (80 + 70 + 90 + 60) / 4 = 300 / 4 = **75.0**

**Classification: Logic error.** Python raises no syntax error (it runs) and no runtime exception (no traceback) - it simply produces the wrong answer.

**Why the output is wrong (one sentence):** `total = s` *overwrites* `total` on every loop instead of adding to it, so only the last score (60) survives and the function returns 60 / 4 = 15.0.

---
## Phase 2 - Trace Table: `average_grade([80, 70, 90, 60])` (buggy version)

| Step | Line / Operation | s | total | return value |
|---|---|---|---|---|
| 1 | `total = 0` | - | 0 | - |
| 2 | `for s in scores` -> `total = s` | 80 | 80 | - |
| 3 | `for s in scores` -> `total = s` | 70 | **70** (80 lost!) | - |
| 4 | `for s in scores` -> `total = s` | 90 | 90 | - |
| 5 | `for s in scores` -> `total = s` | 60 | 60 | - |
| 6 | loop ends (no scores left) | 60 | 60 | - |
| 7 | `len(scores)` | - | 60 | len = 4 |
| 8 | `return total / len(scores)` | - | 60 | **15.0** |

**Faulty step: Step 3.** If `total` were a running sum it would be 80 + 70 = 150 here, but it is 70 - the previous value was thrown away. Correct totals should be 80 -> 150 -> 240 -> 300, returning 75.0.

The cell below prints the same trace automatically so you can check the hand trace against real execution.

In [ ]:
def row(step, op, s, total, ret):
    print(str(step).ljust(6) + op.ljust(30) + str(s).ljust(6) + str(total).ljust(8) + str(ret))

scores = [80, 70, 90, 60]
row('Step', 'Line / Operation', 's', 'total', 'return')
step = 1
total = 0
row(step, 'total = 0', '-', total, '-')
for s in scores:
    step += 1
    total = s   # the buggy line
    row(step, 'for s -> total = s', s, total, '-')
step += 1
row(step, 'loop ends', s, total, '-')
step += 1
row(step, 'len(scores)', '-', total, 'len = ' + str(len(scores)))
step += 1
row(step, 'return total / len(scores)', '-', total, total / len(scores))

---
## Phase 3 - AI Diagnosis & Fix

**Prompt 1 - Classify** - *"I have this Python program. Before fixing anything, tell me: is this a syntax error, a runtime error, or a logic error? Explain why."*
> AI: Logic error. The code is valid Python and runs without an exception, but returns 15.0 instead of the mean 75.0.

**Our classification vs the AI's:** we agreed - logic error. (We classified first, before prompting.)

**Prompt 2 - Diagnose** - *"Now identify the exact line causing the problem and explain what is wrong with it."*
> AI: Line 4, `total = s`. It uses assignment instead of accumulation, so each iteration replaces the running total with the current score. It should be `total = total + s` (or `total += s`).

The AI also pointed out a **second, hidden bug**: `average_grade([])` raises `ZeroDivisionError` because `len([])` is 0.

**Prompt 3 - Fix** - *"Show me the corrected code with a comment explaining each change."* -> applied below.

In [ ]:
def average_grade(scores):
    # FIX 2: guard against an empty list (original crashed with ZeroDivisionError)
    if len(scores) == 0:
        raise ValueError('scores must not be empty')
    total = 0
    for s in scores:
        total = total + s  # FIX 1: accumulate each score (was 'total = s', which overwrote it)
    return total / len(scores)

# Re-run to verify
print(average_grade([80, 70, 90, 60]))   # expected 75.0

---
## Phase 4 - Prove the Fix

### Test Case Design Table (designed BEFORE asking AI)

| # | Input | Expected Output | Category | Rationale |
|---|---|---|---|---|
| 1 | `[80, 70, 90, 60]` | `75.0` | Normal | The instructor's example - buggy code gives 15.0 |
| 2 | `[90, 85, 100]` | `91.6667` (approx) | Normal | Non-whole average; checks real division, not just whole numbers |
| 3 | `[88]` | `88.0` | Boundary | Smallest valid list - one score is its own average |
| 4 | `[75, 75, 75, 75]` | `75.0` | Boundary | All equal - the bug is invisible here, shows why one test is not enough |
| 5 | `[]` | raises `ValueError` | Edge Case | Empty list must give a clear error, not a `ZeroDivisionError` crash |
| 6 | `[0, 100]` | `50.0` | Edge Case | Min and max possible scores; zero must count as a real score |
| 7 | `[100, 100, 100, 0]` | `75.0` | Edge Case | Last score is the outlier - buggy code returns only the last value (0.0) |

### Unit test suite
The tests call `self.func`, so the **same tests** can be run against the fixed function and the original buggy one.

In [ ]:
import unittest

class TestAverageGrade(unittest.TestCase):
    func = staticmethod(average_grade)   # the FIXED function

    def test_1_normal_four_scores(self):
        self.assertEqual(self.func([80, 70, 90, 60]), 75.0)

    def test_2_normal_non_integer_average(self):
        self.assertAlmostEqual(self.func([90, 85, 100]), 91.6667, places=4)

    def test_3_boundary_single_score(self):
        self.assertEqual(self.func([88]), 88.0)

    def test_4_boundary_all_equal(self):
        self.assertEqual(self.func([75, 75, 75, 75]), 75.0)

    def test_5_edge_empty_list(self):
        with self.assertRaises(ValueError):
            self.func([])

    def test_6_edge_min_and_max(self):
        self.assertEqual(self.func([0, 100]), 50.0)

    def test_7_edge_last_score_differs(self):
        self.assertEqual(self.func([100, 100, 100, 0]), 75.0)


def run_tests(test_class):
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(test_class)
    return unittest.TextTestRunner(verbosity=2).run(suite)

print('=== Tests against the FIXED function ===')
run_tests(TestAverageGrade)

**Expected result:** `Ran 7 tests ... OK` - the green bar.

### Proof: the same tests against the ORIGINAL buggy code
At least one test must FAIL on the buggy version - otherwise the tests could not have detected the bug.

In [ ]:
class TestAverageGradeBuggy(TestAverageGrade):
    func = staticmethod(average_grade_buggy)   # the ORIGINAL buggy function

print('=== Same tests against the ORIGINAL buggy function ===')
run_tests(TestAverageGradeBuggy)

**Expected result:** `FAILED (failures=3, errors=1)`

| Test | Buggy result | Outcome |
|---|---|---|
| 1 `[80, 70, 90, 60]` | 15.0 | **FAIL** |
| 2 `[90, 85, 100]` | 33.33 | **FAIL** |
| 3 `[88]` | 88.0 | pass (one score hides the bug) |
| 4 `[75, 75, 75, 75]` | 75.0 | pass (equal scores hide the bug) |
| 5 `[]` | ZeroDivisionError | **ERROR** |
| 6 `[0, 100]` | 50.0 | pass (coincidence: 100 / 2 = 50) |
| 7 `[100, 100, 100, 0]` | 0.0 | **FAIL** |

Tests 3, 4 and 6 pass on BOTH versions - a good reminder that a single "passing" example proves nothing. Tests 1, 2, 5 and 7 are the evidence the bug existed and is now gone.

---
## Phase 5 - AI Tests, Challenge & Gap Analysis

**Prompt 5 - Generate Tests** (run only after our table was complete) and **Prompt 7 - Challenge** (*empty input, None, a string instead of a number, very large values?*). The AI suggested these extra cases:

In [ ]:
class TestAverageGradeAIExtras(unittest.TestCase):
    func = staticmethod(average_grade)

    def test_float_scores(self):
        self.assertAlmostEqual(self.func([85.5, 90.5]), 88.0)

    def test_tuple_input(self):
        self.assertEqual(self.func((80, 70)), 75.0)

    def test_none_input_raises(self):
        with self.assertRaises(TypeError):
            self.func(None)

    def test_string_score_raises(self):
        with self.assertRaises(TypeError):
            self.func([80, '70'])

    def test_large_list(self):
        self.assertEqual(self.func([100] * 100000), 100.0)

    def test_negative_scores_not_rejected(self):
        # Documents a remaining gap: the function does not validate the 0-100 range
        self.assertEqual(self.func([-10, 10]), 0.0)

print('=== AI-suggested extra tests against the FIXED function ===')
run_tests(TestAverageGradeAIExtras)

**Expected result:** `Ran 6 tests ... OK`

### Gap Analysis (Prompt 6) - what each of us missed

| Tests WE designed that the AI did NOT | Tests the AI added that WE missed |
|---|---|
| All-equal scores `[75, 75, 75, 75]` - shows a test that passes on buggy code too, i.e. why one test is never enough | Float scores `[85.5, 90.5]` - grades are not always whole numbers |
| Last score is the outlier `[100, 100, 100, 0]` - targets *exactly* how this bug behaves (only the last value survives) | `None` input -> `TypeError` - bad input from another part of a program |
| Min/max `[0, 100]` - checks 0 is counted as a real score | String inside the list `[80, '70']` - e.g. scores read from a CSV file |
| | Very large list - performance / overflow check |
| | Negative scores - exposed that the function does **not** validate the 0-100 range (a remaining limitation, not fixed here) |

**Does at least one of our tests FAIL on the ORIGINAL buggy code?** [x] Yes

Test 1 (`[80, 70, 90, 60]` -> buggy gives 15.0, expected 75.0) - plus tests 2 and 7 fail and test 5 errors. Because these tests fail on the original and pass on the fixed version, the change from `total = s` to `total = total + s` is what made them pass - that proves the fix, rather than the code just "not crashing".

---
## Debrief Answers (90 seconds)

1. **Did the AI classify correctly and agree with us?** Yes - both said *logic error*: no crash, just a wrong answer (15.0 instead of 75.0). We classified first, then confirmed with the AI.
2. **What did the trace table reveal that the AI diagnosis did not?** The AI named the faulty line, but the trace table *showed* the moment it went wrong: at step 3 `total` became 70 instead of 150 - the earlier score was visibly lost. It also showed why the result is exactly 15.0 (last score 60 / 4).
3. **One test we designed that the AI missed, and one the AI added that we missed:** We added all-equal scores `[75, 75, 75, 75]` (passes even on the buggy code - shows weak tests hide bugs). The AI added a string in the list `[80, '70']` (realistic bad data from files/forms).
4. **Green bar - how do we know the fix is correct, not just non-crashing?** All 7 tests pass on the fixed function, and the same tests fail (3 failures + 1 error) on the original. The buggy code never crashed either, so "not crashing" was never the evidence - matching expected values was.

---
## GitHub Workflow (Claude Code) - ready to paste

**Clone only:**
```
Clone <cases repository URL from instructor> and show me the file for Case C. Do NOT fix, explain, or hint at the bug - I will diagnose the error type myself first.
```

**Commit 1 - the cure:**
```
Fix Case C logic error: average_grade overwrote total (total = s) instead of accumulating it, returning 15.0 not 75.0. Changed to total = total + s and added an empty-list guard (ValueError instead of ZeroDivisionError).
```

**Commit 2 - the proof:**
```
Test suite: 7 cases (normal, boundary, edge) proving Case C fix
```

**Checklist**
- [x] Diagnosed the error type BEFORE AI saw the bug
- [x] Trace table complete; faulty step identified (step 3)
- [ ] Two commits, in order: fix first, tests second (do this in class with the instructor's repo URL)
- [x] Tests pass on the fixed code and at least one fails on the original